In [5]:
# coding: utf-8
import os, re, json, time, random, socket, urllib.parse, urllib.request, urllib.error
from io import BytesIO
from PIL import Image

# ========== 全局参数 ==========
MIN_WIDTH = 300
MIN_HEIGHT = 300
MAX_ASPECT_RATIO = 3
SAVE_ROOT = "./data"

# ========== API盒子 表情包专用接口配置 ==========
# 请替换为你自己的 ID 和 KEY（在 apihz.cn 后台获取）
API_BQB_ID = "10023579"
API_BQB_KEY = "xiaoguangjin12345"
API_BQB_PAGE_SIZE = 20  # 接口上限 20[citation:1]

# ========== 中文关键词 -> 拼音类别名 映射 ==========
KEYWORD_MAP = {
    "奶龙 表情包": "nailong",
    "奶蛙 表情包": "naiwa",
    "水豚噜噜 表情包": "lulu",
    "肥嘟嘟袋鼠 表情包": "daishu",
}

# ========== 类别 -> 首选引擎 ==========
# "api_bqb" 走表情包专用接口，数量不够时自动回退百度
# "baidu" 直接走百度
CATEGORY_ENGINE = {
    "nailong": "api_bqb",
    "naiwa": "api_bqb",
    "lulu": "api_bqb",
    "daishu": "baidu",
}

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8",
    "Accept-Language": "zh-CN,zh;q=0.9,en;q=0.8",
    "Connection": "keep-alive"
}


class Client:
    def __init__(self):
        self.cookies = {}

    def _get_headers(self, referer=None):
        h = HEADERS.copy()
        if self.cookies:
            h["Cookie"] = "; ".join(f"{k}={v}" for k, v in self.cookies.items())
        if referer:
            h["Referer"] = referer
        return h

    def _save_cookies(self, resp_headers):
        for raw in resp_headers.get_all("Set-Cookie") or []:
            first = raw.split(";")[0].strip()
            if "=" in first:
                k, v = first.split("=", 1)
                self.cookies[k.strip()] = v.strip()

    def request(self, url, referer=None, timeout=15):
        req = urllib.request.Request(url, headers=self._get_headers(referer))
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            self._save_cookies(resp.headers)
            return resp.read()


# ========== 解析器 ==========
def parse_baidu(raw):
    try:
        text = raw.decode('utf-8', errors='replace')
        text = re.sub(r'(?<!\\)\\(?!["\\/bfnrtu])', r"\\\\", text)
        data = json.loads(text, strict=False)
        return [i.get("thumbURL") for i in data.get("data", []) if i.get("thumbURL")]
    except:
        return []


def parse_api_bqb(raw):
    """解析 API盒子 表情包接口返回的 JSON"""
    try:
        data = json.loads(raw.decode("utf-8", errors="replace"), strict=False)
        if data.get("code") != 200:
            print(f"  API盒子 返回错误: {data.get('msg', '未知错误')}")
            return [], 0
        return data.get("res", []), int(data.get("maxpage", 0))
    except Exception as e:
        print(f"  API盒子 解析失败: {e}")
        return [], 0


# ========== 命名工具 ==========
def make_name(category, index):
    return f"{category}({index})"


def build_filepath(save_dir, category, index):
    return os.path.join(save_dir, make_name(category, index) + ".jpeg")


# ========== 核心：图片校验与保存 ==========
def validate_and_save(img_data, save_dir, filepath):
    try:
        img = Image.open(BytesIO(img_data))
        w, h = img.size
        if w < MIN_WIDTH or h < MIN_HEIGHT:
            return False, "尺寸太小"

        ratio = max(w, h) / min(w, h)
        if ratio > MAX_ASPECT_RATIO:
            return False, "比例极度异常(长条图)"

        os.makedirs(save_dir, exist_ok=True)
        if img.mode != 'RGB':
            img = img.convert('RGB')
        img.save(filepath, format="JPEG", quality=90)
        return True, "成功"
    except Exception as e:
        return False, f"损坏/非图片格式: {e}"


def download_worker(client, url, save_dir, category, index):
    path = build_filepath(save_dir, category, index)
    # API盒子的图片直接请求，不需要特殊 referer
    referer = "https://image.baidu.com/" if "baidu" in url else \
              "https://res.apihz.cn/" if "apihz" in url else \
              "https://www.bing.com/"
    try:
        time.sleep(random.uniform(0.2, 0.8))
        img_data = client.request(url, referer=referer, timeout=20)

        if len(img_data) < 2000:
            return False, "文件太小"

        success, msg = validate_and_save(img_data, save_dir, path)
        if success:
            print(f"  ✓ 已保存: {make_name(category, index)}.jpeg")
            return True, msg
        else:
            if os.path.exists(path):
                os.unlink(path)
            return False, msg
    except Exception as e:
        return False, f"网络错误: {str(e)[:30]}"


# ========== 专用接口抓取 ==========
def crawl_from_api_bqb(client, keyword, max_count, save_dir, category, saved):
    """从 API盒子 表情包接口抓取，返回新增保存数量"""
    kw_enc = urllib.parse.quote(keyword)
    page = 1
    maxpage = 1

    while saved < max_count and page <= maxpage:
        url = (
            f"https://cn.apihz.cn/api/img/apihzbqb.php"
            f"?id={API_BQB_ID}&key={API_BQB_KEY}"
            f"&type=2&words={kw_enc}&page={page}&limit={API_BQB_PAGE_SIZE}"
        )
        try:
            raw = client.request(url, referer="https://cn.apihz.cn/")
            urls, maxpage = parse_api_bqb(raw)
            if not urls:
                break
            if maxpage == 0:
                maxpage = 8  # 接口异常时给个保守上限
        except Exception as e:
            print(f"  API盒子 请求失败: {e}")
            break

        print(f"  API盒子 第 {page}/{maxpage} 页，返回 {len(urls)} 条")

        for u in urls:
            if saved >= max_count:
                break
            success, msg = download_worker(client, u, save_dir, category, saved + 1)
            if success:
                saved += 1
            elif "损坏" in msg:
                print(f"  ✗ 过滤: {msg}")

        page += 1
        time.sleep(1.0)

    return saved


# ========== 百度兜底抓取 ==========
def crawl_from_baidu(client, keyword, max_count, save_dir, category, saved):
    """从百度抓取，返回新增保存数量"""
    kw_enc = urllib.parse.quote(keyword)
    pn, page = 0, 1

    while saved < max_count and page <= 50:
        url = f"https://image.baidu.com/search/acjson?tn=resultjson_com&word={kw_enc}&pn={pn}&rn=30"
        try:
            raw = client.request(url, referer="https://image.baidu.com/")
            urls = parse_baidu(raw)
        except Exception as e:
            print(f"  百度 获取列表失败: {e}")
            time.sleep(2)
            continue

        if not urls:
            print("  百度 无更多结果")
            break

        for u in urls:
            if saved >= max_count:
                break
            success, msg = download_worker(client, u, save_dir, category, saved + 1)
            if success:
                saved += 1
            elif "损坏" in msg:
                print(f"  ✗ 过滤: {msg}")

        pn += 30
        page += 1
        time.sleep(1.5)

    return saved


# ========== 主流程 ==========
def crawl_images(keyword, max_count):
    query_word = keyword
    category = KEYWORD_MAP.get(keyword, keyword)
    save_dir = os.path.join(SAVE_ROOT, category)
    os.makedirs(save_dir, exist_ok=True)

    client = Client()
    engine = CATEGORY_ENGINE.get(category, "baidu")

    print(f"=== 请求关键词: {query_word} | 命名类别: {category} | 首选引擎: {engine} | 目标: {max_count} 张 ===")

    saved = 0

    # 首选专用接口
    if engine == "api_bqb":
        saved = crawl_from_api_bqb(client, query_word, max_count, save_dir, category, saved)
        if saved < max_count:
            print(f"--- 专用接口获取 {saved} 张，不足 {max_count}，回退百度补充 ---")
            saved = crawl_from_baidu(client, query_word, max_count, save_dir, category, saved)
    else:
        saved = crawl_from_baidu(client, query_word, max_count, save_dir, category, saved)

    print(f"=== 完成！共获取 {saved} 张有效图片，存放于 {save_dir} ===")


In [7]:
from concurrent.futures import ThreadPoolExecutor, as_completed

# 你的关键词列表
keywords = ["奶龙 表情包", "奶蛙 表情包", "水豚噜噜 表情包", "肥嘟嘟袋鼠 表情包"]

def crawl_one(keyword):
    """包装一下 crawl_images，方便在线程池里调用"""
    return crawl_images(keyword, 20)

# 创建线程池，max_workers=4 表示同时跑 4 个任务
with ThreadPoolExecutor(max_workers=4) as executor:
    # 提交任务，建立 future 到关键词的映射，方便出错时知道是哪个词
    future_to_keyword = {
        executor.submit(crawl_one, kw): kw for kw in keywords
    }

    for future in as_completed(future_to_keyword):
        kw = future_to_keyword[future]
        try:
            result = future.result()
            print(f"[完成] {kw} 爬取结束")
        except Exception as e:
            print(f"[失败] {kw} 出错: {e}")

=== 请求关键词: 奶龙 表情包 | 命名类别: nailong | 首选引擎: api_bqb | 目标: 20 张 ===
=== 请求关键词: 水豚噜噜 表情包 | 命名类别: lulu | 首选引擎: api_bqb | 目标: 20 张 ===
=== 请求关键词: 肥嘟嘟袋鼠 表情包 | 命名类别: daishu | 首选引擎: baidu | 目标: 20 张 ===
=== 请求关键词: 奶蛙 表情包 | 命名类别: naiwa | 首选引擎: api_bqb | 目标: 20 张 ===
--- 专用接口获取 0 张，不足 20，回退百度补充 ---
--- 专用接口获取 0 张，不足 20，回退百度补充 ---
--- 专用接口获取 0 张，不足 20，回退百度补充 ---
  百度 无更多结果
=== 完成！共获取 0 张有效图片，存放于 ./data/daishu ===
[完成] 肥嘟嘟袋鼠 表情包 爬取结束
  百度 无更多结果
=== 完成！共获取 0 张有效图片，存放于 ./data/naiwa ===
[完成] 奶蛙 表情包 爬取结束
  百度 无更多结果
=== 完成！共获取 0 张有效图片，存放于 ./data/nailong ===
[完成] 奶龙 表情包 爬取结束
  百度 无更多结果
=== 完成！共获取 0 张有效图片，存放于 ./data/lulu ===
[完成] 水豚噜噜 表情包 爬取结束


In [ ]:
# ========== 自动重编号（人工核查后使用） ==========
def renumber_images(root_dir=SAVE_ROOT):
    if not os.path.exists(root_dir):
        print(f"文件夹 {root_dir} 不存在！")
        return

    print("开始扫描并重编号...")
    
    for folder_name in os.listdir(root_dir):
        folder_path = os.path.join(root_dir, folder_name)
        if not os.path.isdir(folder_path):
            continue

        # 1. 找到该目录下所有符合 拼音(数字).jpeg 的文件
        pattern = re.compile(rf"^{re.escape(folder_name)}\((\d+)\)\.jpeg$")
        valid_files = []
        
        for filename in os.listdir(folder_path):
            match = pattern.match(filename)
            if match:
                num = int(match.group(1))
                valid_files.append((num, filename))

        if not valid_files:
            continue

        # 2. 按原有的数字大小排序
        valid_files.sort(key=lambda x: x[0])

        # 3. 如果编号已经是连续的，就跳过
        expected_nums = list(range(1, len(valid_files) + 1))
        actual_nums = [x[0] for x in valid_files]
        if actual_nums == expected_nums:
            print(f"  [{folder_name}] 编号已连续，无需修改。")
            continue

        # 4. 第一步：全部重命名为临时文件（防止改档冲突覆盖）
        temp_files = []
        for i, (num, filename) in enumerate(valid_files):
            old_path = os.path.join(folder_path, filename)
            temp_name = f"__temp_renumber_{i}__.jpeg"
            temp_path = os.path.join(folder_path, temp_name)
            os.rename(old_path, temp_path)
            temp_files.append(temp_path)

        # 5. 第二步：将临时文件重命名为最终连续编号
        for new_idx, temp_path in enumerate(temp_files, start=1):
            new_name = make_name(folder_name, new_idx) + ".jpeg"
            new_path = os.path.join(folder_path, new_name)
            os.rename(temp_path, new_path)
            
        print(f"  [{folder_name}] 重编号完成，现有 {len(temp_files)} 张图片。")

    print("全部重编号处理完毕！")

renumber_images(SAVE_ROOT)